In [1]:
import numpy as np

class KMeans:
    def __init__(self, n_clusters=3, max_iter=300, tol=1e-4):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.cluster_centers_ = None
        self.labels_ = None
        self.inertia_ = None

    def fit(self, X):
        random_idx = np.random.permutation(X.shape[0])[:self.n_clusters]
        self.cluster_centers_ = X[random_idx]

        for i in range(self.max_iter):
            distances = self._compute_distances(X, self.cluster_centers_)
            labels = np.argmin(distances, axis=1)

            new_centers = np.array([X[labels == j].mean(axis=0) for j in range(self.n_clusters)])

            shift = np.linalg.norm(self.cluster_centers_ - new_centers)
            if shift < self.tol:
                break
            self.cluster_centers_ = new_centers

        self.labels_ = labels
        self.inertia_ = np.sum((X - self.cluster_centers_[labels]) ** 2)

    def predict(self, X):
        distances = self._compute_distances(X, self.cluster_centers_)
        return np.argmin(distances, axis=1)

    def fit_predict(self, X):
        self.fit(X)
        return self.labels_

    def _compute_distances(self, X, centers):
        return np.linalg.norm(X[:, np.newaxis] - centers, axis=2)


In [2]:
import numpy as np

class KMeans:
    def __init__(self, n_clusters=3, max_iter=300, tol=1e-4, n_init=10, init="k-means++"):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.n_init = n_init
        self.init = init

        self.cluster_centers_ = None
        self.labels_ = None
        self.inertia_ = None

    def fit(self, X):
        best_inertia = None
        best_centers = None
        best_labels = None

        for _ in range(self.n_init):
            # Boshlang'ich markazlarni tanlash
            if self.init == "random":
                random_idx = np.random.permutation(X.shape[0])[:self.n_clusters]
                centers = X[random_idx]
            elif self.init == "k-means++":
                centers = self._init_kmeanspp(X)
            else:
                raise ValueError("init faqat 'random' yoki 'k-means++' bo'lishi mumkin")

            for _ in range(self.max_iter):
                distances = self._compute_distances(X, centers)
                labels = np.argmin(distances, axis=1)

                new_centers = []
                for j in range(self.n_clusters):
                    if np.any(labels == j):  # Klaster bo'sh emas
                        new_centers.append(X[labels == j].mean(axis=0))
                    else:  # Agar klaster bo'sh qolsa, eski markazni olib qol
                        new_centers.append(centers[j])
                new_centers = np.array(new_centers)

                shift = np.linalg.norm(centers - new_centers)
                centers = new_centers
                if shift < self.tol:
                    break

            inertia = np.sum((X - centers[labels]) ** 2)

            if best_inertia is None or inertia < best_inertia:
                best_inertia = inertia
                best_centers = centers
                best_labels = labels

        self.cluster_centers_ = best_centers
        self.labels_ = best_labels
        self.inertia_ = best_inertia

    def predict(self, X):
        distances = self._compute_distances(X, self.cluster_centers_)
        return np.argmin(distances, axis=1)

    def fit_predict(self, X):
        self.fit(X)
        return self.labels_

    def _compute_distances(self, X, centers):
        return np.linalg.norm(X[:, np.newaxis] - centers, axis=2)

    def _init_kmeanspp(self, X):
        """K-means++ initialization"""
        n_samples = X.shape[0]
        centers = []
        # Birinchi markazni random tanlash
        first_idx = np.random.randint(0, n_samples)
        centers.append(X[first_idx])

        for _ in range(1, self.n_clusters):
            distances = np.min(self._compute_distances(X, np.array(centers)), axis=1)
            probs = distances ** 2
            probs /= probs.sum()
            next_idx = np.random.choice(n_samples, p=probs)
            centers.append(X[next_idx])

        return np.array(centers)
